In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from tensorflow.keras import regularizers
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

2026-01-26 19:36:10.258559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769456170.479132      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769456170.548961      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769456171.085412      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769456171.085461      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769456171.085464      55 computation_placer.cc:177] computation placer alr

In [2]:
from tensorflow.keras import optimizers
from tensorflow.keras.optimizers import schedules

In [3]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 6.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.1 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=7ae0351bb15b313ac27b1b01b5cd955a4cbc8c4d68ea20be42bfff7c50d392e3
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [4]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential, layers, Model, regularizers
from tensorflow.keras.layers import Input
from sklearn.decomposition import PCA

import os
from lifelines.utils import concordance_index

In [5]:
!pip install SurvivalEVAL

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.1/175.1 kB 6.5 MB/s eta 0:00:00


In [6]:
import random
import csv

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [7]:
from SurvivalEVAL import LifelinesEvaluator
from lifelines import CoxPHFitter

## Creating data



In [8]:
def import_data(path, event="event", time="time-to-event"):

    data = pd.read_csv(path)
    Xs = data.drop(columns=[event, time])
    ys = data[event]
    time = data[time]

    return Xs, ys, time

In [9]:
def lambda_function(x, a):
    """
    Nonlinear function Lambda(x; a) from Bellot & van der Schaar (2019).
    Only the first 5 features are relevant.
    
    x: array of shape (n_samples, n_features)
    a: array-like of length 4 (parameters controlling conditional shift)
    """
    # Ensure we have at least 5 features
    x1, x2, x3, x4, x5 = x[:, 0], x[:, 1], x[:, 2], x[:, 3], x[:, 4]
    
    return (10 * a[0] * x1 * (x1 - 1) + #* (x1 + 1) * (x1 - 2) +
            5  * a[1] * (x2 ** 2) -
            2  * a[2] * np.log(x3 + 3.0) -
            a[3] * ((x4 - 1) ** 2))

    # return 5 * a[0] * x1 + 4 * a[1] * (x2) - 3  * a[2] * x3 + 2 * a[3] * x4**2

def generate_survival_data(n, d_features=10, delta=1.0, conditional_shift=0.0,  
                           censoring_rate=0.2, shift_type = 'marginal', data_type='target', seed=42, save_data=True):
    """
    Generate synthetic survival data.
    
    Parameters
    ----------
    n : int
        Number of samples
    d_features : int
        Number of features in the target domain (default 10, 5 informative, 5 noise)
    delta : float in [0,1]
        Proportion of source samples overlapping with target support [-1,1]
    conditional_shift : float
        Variance factor for Gaussian perturbation of a (conditional shift)
    heterogeneous : bool
        If True, simulate heterogeneous source (different feature size)
    add_noise_dim : int
        Number of extra pure noise dimensions in source (if heterogeneous)
    drop_informative : int
        Number of informative features to drop in source (if heterogeneous)
    censoring_rate : float
        Proportion of censoring (approx)
    seed : int
        Random seed
    
    Returns
    -------
    X : ndarray, shape (n, d_features)
        Covariates
    T_obs : ndarray, shape (n,)
        Observed survival/censoring times
    E : ndarray, shape (n,)
        Event indicator (1=event, 0=censored)
    """
    rng = np.random.default_rng(seed)

    # --- Step 1: Generate features ---
    # target support is [-1, 1]
    overlap_size = int(n * delta)
    outside_size = n - overlap_size

    # arr = np.arange(0, 200, 20) 

    # overlapping part
    X_overlap = rng.uniform(-1, 1, size=(overlap_size, d_features))

    # arr_out = np.arange(-20, 270, 30)
    
    # outside part
    X_outside = rng.uniform(-1.5, 2.5, size=(outside_size, d_features))

    mask = (X_outside[:, 0] < -1) | (X_outside[:, 0] > 1)
    X_outside = X_outside[mask]  # ensure outside support

    if len(X_outside) < outside_size:
        # resample if too few
        extra = outside_size - len(X_outside)
        extra_samples = rng.uniform(-1.5, 2.5, size=(extra, d_features))
        X_outside = np.vstack([X_outside, extra_samples])
    
    X = np.vstack([X_overlap, X_outside])
    rng.shuffle(X)

    # --- Step 2: Conditional shift ---
    # a = [1,1,1,1] for target; add noise for source
    a = np.ones((d_features, len(X)))

    # print(a.shape)

    # --- Step 3: Generate survival times ---
    Lambda = lambda_function(X, a)

    # Weibull
    k = 2.0
    scale = np.exp(-Lambda)
    # print('scale shape', scale.shape)
    # print('Lambda', Lambda[:5])
    # print('Max Lambda', Lambda.max())
    # print('Min Lambda', Lambda.min())
    # Clip to a reasonable range, e.g. [1, 10]
    # scale = np.clip(scale, 1, 100)
    T = rng.weibull(a=k, size=len(X)) * scale
    # T_ = rng.weibull(a=k, size=len(X))
    # print('T', T[:5])
    # print('T shape', T.shape)
    # print('T_', T_[:5])
    # print('T_ shape', T_.shape)

    # print(scale)

    # --- Step 4: Apply censoring ---
    # α=2 → 50% censored
    # α=5, equivalento to censoring_rate = 0.2 → 20% censored
    # α=10, equivalento to censoring_rate = 0.1 → 10% censored

    C = rng.uniform(0, (1/censoring_rate)*T)   
    T_obs = np.minimum(T, C)
    E = (T <= C).astype(int)

    df = pd.DataFrame(X, columns=[f"x{i+1}" for i in range(X.shape[1])])
    df["time-to-event"] = T_obs
    df["event"] = E

    if save_data:

        # Ensure directory exists
        output_dir = "datasets/simulated_data"
        os.makedirs(output_dir, exist_ok=True)

        # Save to CSV
        output_path = os.path.join(output_dir, f"{shift_type}_shift_{data_type}.csv")
        df.to_csv(output_path, index=False)

        plot_time_to_event_save(T_obs, E, bins=30, output_dir="datasets/simulated_data/plots", 
                                shift_type=shift_type, data_type=data_type)
        

    return X, T_obs, E


def plot_time_to_event_save(T, E, bins=30, output_dir="datasets/simulated_data/plots", 
                            shift_type='marginal', data_type='target'):
    """
    Plot histogram of time-to-event values and separate histograms for events vs censored,
    then save the figures into the specified output directory.
    
    Parameters
    ----------
    T : array-like
        Time-to-event values
    E : array-like
        Event indicator (1=event, 0=censored)
    bins : int
        Number of bins for histograms
    output_dir : str
        Directory path to save the plots
    """
    os.makedirs(output_dir, exist_ok=True)


    # --- Separate histogram: events vs censored ---
    plt.figure(figsize=(7,5))
    plt.hist(np.array(T)[np.array(E)==1], bins=bins, alpha=0.7, label="Events", color="steelblue", edgecolor="black")
    plt.hist(np.array(T)[np.array(E)==0], bins=bins, alpha=0.7, label="Censored", color="orange", edgecolor="black")
    plt.xlabel("Time-to-event")
    plt.ylabel("Frequency")
    plt.title("Events vs. Censored Distribution")
    plt.legend()
    sep_hist_path = os.path.join(output_dir, f"{shift_type}_shift_{data_type}_histogram_events_vs_censored.png")
    plt.savefig(sep_hist_path, dpi=300, bbox_inches="tight")
    plt.close()

In [10]:
def cox_feature_selection(df, penalizer=0.1, threshold=0.01, duration_col="time", event_col="cens"):
    
    model = CoxPHFitter(penalizer=penalizer).fit(df, duration_col=duration_col, event_col=event_col)
    
    # Get the coefficients of the features
    coefficients = model.params_

    abs_coefs = np.abs(coefficients)

    print(abs_coefs)

    # Keep features with |coef| > threshold
    selected_features = abs_coefs[abs_coefs > threshold].index.tolist()

    # print("Selected features:", selected_features)

    return selected_features

In [11]:
def selected_(Xs, Xt, ys, yt, Ts, Tt, threshold=1e-3, duration_col="time", event_col="cens", specs_=True):

    def convert_dataframe(X_train_source, y_train_source, time_source_train):

        # Convert numpy arrays to DataFrames
        X_train_df = pd.DataFrame(X_train_source, columns=[f"x{i}" for i in range(X_train_source.shape[1])])
        
        # --- Training set ---
        train_df_source = pd.concat(
            [X_train_df.reset_index(drop=True),
             pd.Series(time_source_train.reshape(-1), name="time").reset_index(drop=True),
             pd.Series(y_train_source.reshape(-1), name="cens").reset_index(drop=True)],
            axis=1
        )

        return train_df_source

    def prepare_arrays(df_selected, duration_col="time", event_col="cens"):
        """
        Convert selected dataframe into numpy arrays for Cox model or neural network.
        """
        # Separate features and targets
        X = df_selected.drop(columns=[duration_col, event_col]).to_numpy()
        y_time = df_selected[duration_col].to_numpy()
        y_event = df_selected[event_col].to_numpy()
    
        return X, y_time, y_event

        
    X_train, _, y_train, _, time_train, _ = train_test_split(Xs, ys, Ts, test_size=0.3, random_state=42)

    dfs = convert_dataframe(X_train, y_train, time_train)
    
    source_selected_features = cox_feature_selection(dfs, penalizer=0.1, threshold=1e-2, duration_col=duration_col, event_col=event_col)

    dfs_ = convert_dataframe(Xs, ys, Ts)
    dfs_selected = dfs_[source_selected_features + [duration_col, event_col]]

    X_source, T_source, y_source = prepare_arrays(dfs_selected)

    
    ## Target
    Xt_train, _, yt_train, _, timet_train, _ = train_test_split(Xt, yt, Tt, test_size=0.3, random_state=42)

    dft = convert_dataframe(Xt_train, yt_train, timet_train)

    target_selected_features = cox_feature_selection(dft, penalizer=0.1, threshold=threshold, duration_col=duration_col, event_col=event_col)

    dft_ = convert_dataframe(Xt, yt, Tt)

    dft_selected = dft_[target_selected_features + [duration_col, event_col]]

    X_target, T_target, y_target = prepare_arrays(dft_selected)

    if specs_:
        print("Source Data Statistics: ")
        print(f"X_source shape: {X_source.shape}")
        print(f"y_source sum: {y_source.sum()}")
        print(f"min_max source time: {np.round(T_source.min(),2)}, {np.round(T_source.max(),2)}")
        print(" ")
        print("Target Data Statistics: ")
        print(f"X_target shape: {X_target.shape}")
        print(f"y_target sum: {y_target.sum()}")
        print(f"min_max source time: {np.round(Tt.min(),2)}, {np.round(Tt.max(),2)}")
        

    return {"X_source":X_source,
            "y_source":y_source,
            "Time_source":T_source,
            "X_target":X_target,
            "y_target":y_target,
            "Time_target":T_target,}

In [12]:
def assign_unlabel(y_train, time_target_train, unlabeled_fraction=0):

    np.random.seed(42)
    
    
    # Set fraction of target training samples to be unlabeled
    n_target = len(y_train)
    n_unlabeled = int(unlabeled_fraction * n_target)
    
    # Randomly choose indices to make unlabeled
    unlabeled_indices = np.random.choice(n_target, n_unlabeled, replace=False)
    
    y_train = np.array(y_train, dtype=float)
    time_target_train = np.array(time_target_train, dtype=float)
    
    # Mask with np.nan
    y_train[unlabeled_indices] = np.nan
    time_target_train[unlabeled_indices] = np.nan

    # Count how many labels are missing (i.e., unlabeled)
    n_unlabeled_samples = np.isnan(y_train).sum()

    
    print(f"\nNumber of unlabeled samples: {n_unlabeled_samples}")

    n_labeled_samples = np.sum(~np.isnan(y_train))
    print(f"Number of labeled samples: {n_labeled_samples}")

    return y_train, time_target_train

In [13]:
import numpy as np

def oversample_target(X_target, y_target, T_target, n_source, seed=42):
    """
    Oversample target set to match the source size, handling
    fully labeled, partially labeled, or fully unlabeled cases.

    Parameters
    ----------
    X_target : np.ndarray
        Target covariates/features, shape (n_t, d).
    y_target : np.ndarray
        Target labels, shape (n_t,). Can contain np.nan for unlabeled.
    T_target : np.ndarray
        Target times, shape (n_t,). Can contain np.nan for unlabeled.
    n_source : int
        Desired size (same as source sample size).
    seed : int or None
        Random seed.

    Returns
    -------
    X_resampled, y_resampled, T_resampled : np.ndarray
        Oversampled target set with size = n_source.
    """
    rng = np.random.default_rng(seed)

    n_target = len(y_target)

    # Case 1: already the right size
    if n_target == n_source:
        return X_target, y_target, T_target

    # Case 2: smaller → oversample with replacement
    if n_target < n_source:
        idx = rng.choice(n_target, n_source, replace=True)
        return X_target[idx], y_target[idx], T_target[idx]

    # Case 3: larger → downsample without replacement
    idx = rng.choice(n_target, n_source, replace=False)
    return X_target[idx], y_target[idx], T_target[idx]

## Domain Alignment

In [14]:
def params_(same_unit = 200, A_unit=200, l2_reg=1e-1):


    Ps = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_P_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="source_P_BN1")

    ])
    
    Pt = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="target_P_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="target_P_BN1")
    ])
    

    # Source B network
    Bs = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_B_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="source_B_BN1")

    ])
    
    # Target B network
    Bt = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="target_B_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="target_B_BN1")

    ])

    A = Sequential([
        layers.Dense(A_unit, use_bias=False, activation=None, name="SharedLayer_A_Dense"),
        # layers.BatchNormalization(name="SharedLayer_A_BN")
    ])
    
    D = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="SharedLayer_D_Dense"),
        # layers.BatchNormalization(name="SharedLayer_D_BN")
    ])

    phiN = tf.keras.Sequential([
        layers.Dense(32, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_N_dense"),
        # layers.BatchNormalization(name="phi_N_batch_norm"),  # BatchNorm layer added
    ])

    phiM = tf.keras.Sequential([
        layers.Dense(64, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_M_dense"),
        # layers.BatchNormalization(name="phi_M_batch_norm"),  # BatchNorm layer added
    ])


    return D, A, Bs, Bt, Ps, Pt, phiN, phiM


In [15]:
def params_1layer(same_unit = 16, A_unit=16, l2_reg=1e-3):


    Ps = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_P_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="source_P_BN1"),
        layers.ReLU(name="source_P_ReLU1"),

    ])
    
    Pt = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="target_P_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="target_P_BN1"),
        layers.ReLU(name="source_P_ReLU1"),

    ])
    

    # Source B network
    Bs = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_B_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="source_B_BN1"),
        layers.ReLU(name="source_B_ReLU1"),

    ])
    
    # Target B network
    Bt = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="target_B_Dense1",
        kernel_regularizer=regularizers.l2(l2_reg)),
        # layers.BatchNormalization(name="target_B_BN1"),
        layers.ReLU(name="target_B_ReLU1"),

    ])

    A = Sequential([
        layers.Dense(A_unit, use_bias=False, activation=None, name="SharedLayer_A_Dense"),
        #layers.BatchNormalization(name="SharedLayer_A_BN")
    ])
    
    D = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="SharedLayer_D_Dense"),
        #layers.BatchNormalization(name="SharedLayer_D_BN")
    ])

    phiN = tf.keras.Sequential([
        layers.Dense(32, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_N_dense"),
        #layers.BatchNormalization(name="phi_N_batch_norm"),  # BatchNorm layer added
    ])

    phiM = tf.keras.Sequential([
        layers.Dense(64, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_M_dense"),
        #layers.BatchNormalization(name="phi_M_batch_norm"),  # BatchNorm layer added
    ])


    return D, A, Bs, Bt, Ps, Pt, phiN, phiM


In [16]:
def params_2layer(different_unit = 8, same_unit = 4, A_unit=50):


    Ps = Sequential([
        layers.Dense(different_unit, use_bias=False, activation=None, name="source_P_Dense1"),
        #layers.BatchNormalization(name="source_P_BN1"),
        layers.ReLU(name="source_P_ReLU1"),

        # Second hidden layer
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_P_Dense2"),
        #layers.BatchNormalization(name="source_P_BN2"),
        layers.ReLU(name="source_P_ReLU2")
    ])
    
    Pt = Sequential([
        layers.Dense(different_unit, use_bias=False, activation=None, name="target_P_Dense1"),
        #layers.BatchNormalization(name="target_P_BN1"),
        layers.ReLU(name="source_P_ReLU1"),

        layers.Dense(same_unit, use_bias=False, activation=None, name="target_P_Dense2"),
        #layers.BatchNormalization(name="target_P_BN2"),
        layers.ReLU(name="target_P_ReLU2")
    ])
    

    # Source B network
    Bs = Sequential([
        layers.Dense(different_unit, use_bias=False, activation=None, name="source_B_Dense1"),
        #layers.BatchNormalization(name="source_B_BN1"),
        layers.ReLU(name="source_B_ReLU1"),
    
        layers.Dense(same_unit, use_bias=False, activation=None, name="source_B_Dense2"),
        #layers.BatchNormalization(name="source_B_BN2"),
        layers.ReLU(name="source_B_ReLU2")
    ])
    
    # Target B network
    Bt = Sequential([
        layers.Dense(different_unit, use_bias=False, activation=None, name="target_B_Dense1"),
        # layers.BatchNormalization(name="target_B_BN1"),
        layers.ReLU(name="target_B_ReLU1"),
    
        layers.Dense(same_unit, use_bias=False, activation=None, name="target_B_Dense2"),
        #layers.BatchNormalization(name="target_B_BN2"),
        layers.ReLU(name="target_B_ReLU2")
    ])

    A = Sequential([
        layers.Dense(A_unit, use_bias=False, activation=None, name="SharedLayer_A_Dense"),
        layers.BatchNormalization(name="SharedLayer_A_BN")
    ])
    
    D = Sequential([
        layers.Dense(same_unit, use_bias=False, activation=None, name="SharedLayer_D_Dense"),
        layers.BatchNormalization(name="SharedLayer_D_BN")
    ])

    phiN = tf.keras.Sequential([
        layers.Dense(32, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_N_dense"),
        layers.BatchNormalization(name="phi_N_batch_norm"),  # BatchNorm layer added
    ])

    phiM = tf.keras.Sequential([
        layers.Dense(64, use_bias=False, activation=None, kernel_initializer='he_normal', name="phi_M_dense"),
        layers.BatchNormalization(name="phi_M_batch_norm"),  # BatchNorm layer added
    ])

    return D, A, Bs, Bt, Ps, Pt, phiN, phiM


## Survival neural nets model

In [17]:
def build_surv_block(
    hidden_units=[32, 16],
    activation='relu',
    dropout_rate=0.0,
    reg_type='l1',
    reg_strength=0.01,
    use_batchnorm=True,
    name_prefix="surv",
    print_specs=False):
    
    """
    Returns a stackable tf.keras.Sequential model (no input_shape),
    along with a string summarizing the configuration.

    Args:
        hidden_units (list[int]): Units per hidden layer.
        activation (str): Activation function after each layer.
        dropout_rate (float): Dropout rate.
        reg_type (str): 'l1', 'l2', or None.
        reg_strength (float): λ for regularization.
        use_batchnorm (bool): Whether to include batchnorm.
        name_prefix (str): Prefix for layer names.
        print_specs (bool): Whether to print the config string.

    Returns:
        model (tf.keras.Sequential), config_summary (str)
    """
    # Select regularizer
    if reg_type == 'l1':
        kernel_reg = regularizers.l1(reg_strength)
    elif reg_type == 'l2':
        kernel_reg = regularizers.l2(reg_strength)
    elif reg_type == 'l1_l2':
        kernel_reg = regularizers.l1_l2(l1=reg_strength, l2=reg_strength)
    else:
        kernel_reg = None

    model = tf.keras.Sequential(name=f"{name_prefix}_block")

    for idx, units in enumerate(hidden_units):

        if units <= 0:
            continue  

        model.add(layers.Dense(
            units=units,
            activation=None,
            kernel_regularizer=kernel_reg,
            use_bias=not use_batchnorm,
            name=f"{name_prefix}_dense_{idx+1}"
        ))

        if use_batchnorm:
            model.add(layers.BatchNormalization(name=f"{name_prefix}_bn_{idx+1}"))
            
        model.add(layers.Activation(activation, name=f"{name_prefix}_act_{idx+1}"))

        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout_{idx+1}"))

    model.add(layers.Dense(1, activation=None, kernel_regularizer=kernel_reg, name=f"{name_prefix}_output"))

    # Create configuration summary string
    config_summary = (
        "\n>>> Survival model configuration:\n"
        f"  • input_shape    = (unspecified, for stacking only)\n"
        f"  • hidden_units   = {hidden_units}\n"
        f"  • activation     = {activation}\n"
        f"  • dropout_rate   = {dropout_rate}\n"
        f"  • regularizer    = {reg_type} (λ={reg_strength})\n"
        f"  • batch_norm     = {use_batchnorm}\n"
        ">>> End of configuration\n"
    )

    if print_specs:
        print(config_summary)
        
    return model, config_summary

## Losses

In [18]:
def gaussian_kernel_tf(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    Computes a multi-scale Gaussian RBF kernel among all samples from `source` and `target`.
    source: shape (batch_size, feature_dim)
    target: shape (batch_size, feature_dim)
    """
    # Concatenate source & target
    total = tf.concat([source, target], axis=0)  # shape (batch_size_s + batch_size_t, feature_dim)
    n_samples = tf.shape(total)[0]

    # Expand dims to compute pairwise distances (L2)
    total0 = tf.expand_dims(total, axis=0)
    total1 = tf.expand_dims(total, axis=1)

    # L2_distance shape = (n_samples, n_samples)
    L2_distance = tf.reduce_sum((total0 - total1) ** 2, axis=2)

    if fix_sigma is not None:
        bandwidth = fix_sigma
    else:
        # Compute average L2 distances, ignoring diagonal
        n_samples_f = tf.cast(n_samples, tf.float32)
        bandwidth = tf.reduce_sum(L2_distance) / (n_samples_f**2 - n_samples_f)

    # Adjust bandwidths for multi-kernel
    bandwidth = bandwidth / (kernel_mul ** (kernel_num // 2))
    bandwidths = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    # Compute each Gaussian kernel
    kernel_vals = []
    for bw in bandwidths:
        # Stabilize exponential computation
        kernel_vals.append(tf.exp(-tf.clip_by_value(L2_distance / (2*bw), -1e2, 1e2)))

    final_kernel = tf.add_n(kernel_vals)

    return final_kernel

In [19]:
def count_classes(y):
    # Convert TensorFlow tensor to NumPy array
    if isinstance(y, tf.Tensor):
        y = y.numpy()

    # Optionally cast to int for class label clarity
    y = y.astype(int)

    unique_classes, class_counts = np.unique(y, return_counts=True)

    print("Set class distribution:")
    for cls, count in zip(unique_classes, class_counts):
        print(f"Class {cls}: {count} samples")

In [20]:
def prepare_labels_and_gather(labels, predictions, time, sort_descending=True):
    """
    - Ensures labels, predictions, and time are 1D float32 tensors
    - Sorts by time and applies the same order to all
    """

    # Convert and cast labels
    labels = tf.convert_to_tensor(labels)
    if labels.dtype != tf.float32:
        labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, [-1])

    # Convert and cast predictions
    predictions = tf.convert_to_tensor(predictions)
    if predictions.dtype != tf.float32:
        predictions = tf.cast(predictions, tf.float32)
    predictions = tf.reshape(predictions, [-1])

    # Convert and cast time
    time = tf.convert_to_tensor(time)
    if time.dtype != tf.float32:
        time = tf.cast(time, tf.float32)
    time = tf.reshape(time, [-1])

    # Sort
    direction = 'DESCENDING' if sort_descending else 'ASCENDING'
    order = tf.argsort(time, direction=direction)

    # Apply order
    sorted_labels = tf.gather(labels, order)
    sorted_preds = tf.gather(predictions, order)
    sorted_time = tf.gather(time, order)

    return sorted_labels, sorted_preds

In [21]:
def cox_ph_loss_batch(label, time, y_pred):
    """
    Cox Partial Likelihood Loss for mini-batch training (with numerical stability).
    
    Args:
        event: Tensor of shape (batch_size,) with 1 = event occurred, 0 = censored
        time: Tensor of shape (batch_size,) with time-to-event or censoring
        y_pred: Tensor of shape (batch_size, 1) with log-risk scores from model output

    Returns:
        Negative partial log-likelihood loss (scalar)
    """

    def to_tensor_column(x, dtype=tf.float32):
        # Convert Pandas Series or NumPy array to NumPy
        if isinstance(x, pd.Series):
            x = x.values
        elif tf.is_tensor(x):
            return tf.reshape(tf.cast(x, dtype=dtype), [-1, 1])
        
        # Now x is a NumPy array
        return tf.convert_to_tensor(np.reshape(x, (-1, 1)), dtype=dtype)

    time = to_tensor_column(time, dtype=tf.float32)

    #print(label)
    #print(time)
    #print(y_pred)
    
    # Make sure label is shaped properly
    label = tf.reshape(label, [-1])  # flatten to shape (batch_size,)

    #print('Before argsort with gather in label')
    #count_classes(label)
    #print(' ')
    
    event, pred = prepare_labels_and_gather(
        labels=label,
        predictions=y_pred,
        time=time
    )
    
    #print('After argsort with gather in label')
    #count_classes(event)
    #print(' ')

    # --- Handle extreme predictions ---
    eps = 1e-5
    gamma = tf.reduce_max(pred)
    stable_pred = tf.clip_by_value(pred - gamma, -40.0, 40.0)  # avoid overflow
    exp_preds = tf.exp(stable_pred)

    # --- Numerically stable log-cumsum ---
    cumsum = tf.cumsum(exp_preds)
    log_cumsum = tf.math.log(tf.clip_by_value(cumsum, eps, 1e9)) + gamma

    log_likelihood = pred - log_cumsum

    # Step 4: Keep only uncensored events
    uncensored = tf.boolean_mask(log_likelihood, event > 0)

    # Step 5: Return negative mean log-likelihood
    mean_log_l = -tf.reduce_mean(uncensored)
    
    return mean_log_l

In [22]:
class UnitNormClipper:
    def __call__(self, layer):
        # Ensure the layer has weights
        if hasattr(layer, 'kernel'):
            weights = layer.kernel
            # Compute the norms for each weight vector (column-wise for dense layers)
            norms = tf.norm(weights, axis=0, keepdims=True)  # Change axis if necessary for your layer
            # Normalize weights to unit norm
            layer.kernel.assign(weights / (norms + 1e-8))  # Add small epsilon to prevent division by zero

In [23]:
def save_weights(classifier_, model_type, weights_save_dir):

    # # Directory to save weights
    # weights_save_dir = "./model_weights"
    # os.makedirs(weights_save_dir, exist_ok=True)
    
    # Save the weights of each model
    classifier_.save_weights(os.path.join(weights_save_dir, f"{model_type}_classifier_.weights.h5"))
    
    # print(f"Weights saved to {weights_save_dir}")

In [24]:
def survival_metrics(model, X, durations, events, data_type=""):
    """
    Compute concordance index, ignoring samples with np.nan in durations or events.
    If no valid labeled samples or admissible pairs exist, return np.nan.

    Args:
        model: tf.keras model
        X: input features
        durations: time-to-event (can include np.nan)
        events: event indicator (1 = event, 0 = censor, np.nan = unlabeled)
        data_type: optional string label
    Returns:
        adjusted concordance index (float or np.nan)
    """
    durations = np.asarray(durations).squeeze()
    events = np.asarray(events).squeeze()

    # Mask valid (non-NaN) entries
    valid_mask = ~np.isnan(durations) & ~np.isnan(events)

    if np.sum(valid_mask) < 2:
        #print(f"[Warning] Too few valid samples in {data_type}. Skipping C-index.")
        return 0.0

    X_valid = X[valid_mask]
    durations_valid = durations[valid_mask]
    events_valid = events[valid_mask]

    if isinstance(X_valid, pd.DataFrame):
        X_valid = X_valid.to_numpy()

    X_valid = tf.convert_to_tensor(X_valid, dtype=tf.float32)
    risk_scores = model(X_valid).numpy().squeeze()

    try:
        cindex = concordance_index(durations_valid, -risk_scores, events_valid)
        return 1.0 - cindex if cindex < 0.5 else cindex
    
    except ZeroDivisionError:
        # print(f"[Warning] No admissible pairs for {data_type}. Returning np.nan.")
        return np.nan


In [25]:
class StratifiedBatchGenerator:
    def __init__(self, X_source, y_source, time_source,
                 X_target, y_target, time_target,
                 batch_size=1024, shuffle=True, seed=42):
        
        self.X_source = X_source
        self.y_source = y_source
        self.time_source = time_source

        self.X_target = X_target
        self.y_target = y_target
        self.time_target = time_target

        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.rng = np.random.default_rng(seed) if seed is not None else np.random

        self.n_samples = len(X_target)
        self.indices = np.arange(self.n_samples)

        # Check if target has any labels at all
        self.has_labels = not np.all(np.isnan(y_target))

        if self.has_labels:
            # Ensure at least one uncensored sample exists
            uncensored_exists = np.any(y_target == 1)
            assert uncensored_exists, "There must be at least one uncensored sample in the target domain."

        if shuffle:
            self.rng.shuffle(self.indices)

        self.num_batches = int(np.ceil(self.n_samples / self.batch_size))
        self.i = 0

    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.num_batches:
            raise StopIteration

        start = self.i * self.batch_size
        end = min((self.i + 1) * self.batch_size, self.n_samples)
        idx = self.indices[start:end]

        # Target batch
        batch_target = self.X_target[idx]
        y_batch_target = self.y_target[idx]
        time_batch_target = self.time_target[idx]

        if self.has_labels:
            # If no uncensored in this batch, inject one
            if not np.any(y_batch_target == 1):
                all_uncensored_idx = np.where(self.y_target == 1)[0]
                substitute_idx = self.rng.choice(all_uncensored_idx, size=1)
                idx[-1] = substitute_idx[0]
                batch_target = self.X_target[idx]
                y_batch_target = self.y_target[idx]
                time_batch_target = self.time_target[idx]
        else:
            # Fully unlabeled target → replace with NaNs
            y_batch_target = np.full(len(idx), np.nan)
            time_batch_target = np.full(len(idx), np.nan)

        # Source batch
        source_idx = self.rng.choice(len(self.X_source), size=len(idx), replace=False)
        batch_source = self.X_source[source_idx]
        y_batch_source = self.y_source[source_idx]
        time_batch_source = self.time_source[source_idx]

        self.i += 1
        return (batch_source, y_batch_source, time_batch_source,
                batch_target, y_batch_target, time_batch_target)


In [26]:
def pipeline_MMD(batch_source, batch_target, y_batch_source, y_batch_target, all_target_X,
                D, A, Bs, Bt, Ps, Pt,
                phiN, phiM,
                hc, 
                time_batch_source, time_batch_target, 
                source_weight = 0.5,
                target_weight = 1,
                coef_1 = 1, # Mesma ordem de grandeza
                coef_2 = 1e-8,
                coef_3 = 1e-8,
                learning_rate_dict=0.01, momentum_dict=0.99,
                max_coef = 1, learning_rate_M=0.01, momentum_M=0.99,
                min_coef = 1, learning_rate_N=0.01, momentum_N=0.99):
    """
    Training step for complete pipeline including dictionary, adversarial, and classification losses.
    """

    lr_schedule_M = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,  # start LR
    decay_steps=800,            # every 1000 steps
    decay_rate=1e-2,             # multiply LR by 0.96
    staircase=True               # discrete drops instead of continuous
    )

    lr_schedule_N = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,  # start LR
    decay_steps=800,            # every 1000 steps
    decay_rate=1e-2,             # multiply LR by 0.96
    staircase=True               # discrete drops instead of continuous
    )

    optimizer_class_dict = tf.keras.optimizers.SGD(learning_rate=learning_rate_dict, momentum = momentum_dict)
    optimizer_phiM = tf.keras.optimizers.SGD(learning_rate=lr_schedule_M, momentum = momentum_M)
    optimizer_phiN = tf.keras.optimizers.SGD(learning_rate=lr_schedule_N, momentum = momentum_N)

    unit_norm_clipper = UnitNormClipper()
    
    with tf.GradientTape(persistent=True) as tape:

        D.trainable = True
        A.trainable = True
        Bs.trainable = True
        Bt.trainable = True
        Ps.trainable = True
        Pt.trainable = True
        hc.trainable = True
        phiN.trainable = True
        phiM.trainable = False
        
        # DICTIONARY: Reconstruction and Regularization ##########################################
        
        PsXs_ = Ps(batch_source)
        PtXt_ = Pt(batch_target)

        # print(f"PsXs_ shape: {PsXs_.shape}")
        # print(f"PsXt_ shape: {PtXt_.shape}")

        ABsXs_ = A(Bs(batch_source))
        DABsXs_ = D(ABsXs_)

        # print(f"ABsXs_ shape: {ABsXs_.shape}")
        # print(f"DABsXs_ shape: {DABsXs_.shape}")

        ABtXt_ = A(Bt(batch_target))
        DABtXt_ = D(ABtXt_)

        # print(f"ABtXt_ shape: {ABtXt_.shape}")
        # print(f"DABtXt_ shape: {DABtXt_.shape}")

        recon_loss_source = tf.sqrt(tf.reduce_mean(tf.square(PsXs_ - DABsXs_)))
        recon_loss_target = tf.sqrt(tf.reduce_mean(tf.square(PtXt_ - DABtXt_)))

        reg_term_source = tf.reduce_mean(tf.abs(ABsXs_))
        reg_term_target = tf.reduce_mean(tf.abs(ABtXt_))

        norm_penalty = - tf.reduce_sum(1 - tf.norm(D.trainable_variables[0], ord=2, axis=0))
        norm_penalty_A = - tf.reduce_sum(1 - tf.norm(A.trainable_variables[0], ord=2, axis=0))

        ortho_penalty = - (
            tf.reduce_sum(tf.square(tf.eye(Ps.layers[0].units) - tf.matmul(tf.transpose(Ps.layers[0].trainable_variables[0]), Ps.layers[0].trainable_variables[0]))) +
            tf.reduce_sum(tf.square(tf.eye(Pt.layers[0].units) - tf.matmul(tf.transpose(Pt.layers[0].trainable_variables[0]), Pt.layers[0].trainable_variables[0])))
        )

        ortho_penalty_b = - (
            tf.reduce_sum(tf.square(tf.eye(Bs.layers[0].units) - tf.matmul(tf.transpose(Bs.layers[0].trainable_variables[0]), Bs.layers[0].trainable_variables[0]))) +
            tf.reduce_sum(tf.square(tf.eye(Bt.layers[0].units) - tf.matmul(tf.transpose(Bt.layers[0].trainable_variables[0]), Bt.layers[0].trainable_variables[0])))
        )

        dict_loss = recon_loss_source + recon_loss_target

        phiN_s = phiN(ABsXs_)
        phiM_s = phiM(phiN_s)

        phiN_t = phiN(ABtXt_)
        phiM_t = phiM(phiN_t)

        # print(phiN_s.shape)
        # print(phiN_t.shape)
 
        hc_s = hc(phiN_s)
        hc_t = hc(phiN_t)

        loss_surv_source = cox_ph_loss_batch(y_batch_source, time_batch_source, hc_s)

        y_batch_target_ = y_batch_target.reshape(-1)
        
        # Identify which target samples are labeled
        valid_mask_target = ~tf.math.is_nan(y_batch_target_) & ~tf.math.is_nan(time_batch_target)
        
        #print(valid_mask_target)
        
        # Apply mask only for supervised loss computation
        if tf.reduce_any(valid_mask_target):

            # print('y_batch_target', y_batch_target.shape)
            # print('valid_mask_target', valid_mask_target.shape)
            
            y_target_valid = tf.boolean_mask(y_batch_target, valid_mask_target)
            t_target_valid = tf.boolean_mask(time_batch_target, valid_mask_target)
            hc_t_valid = tf.boolean_mask(hc_t, valid_mask_target)
        
            loss_surv_target = cox_ph_loss_batch(y_target_valid, t_target_valid, hc_t_valid)
            
        else:
            loss_surv_target = tf.constant(0.0, dtype=tf.float32)

        
        surv_loss = source_weight*loss_surv_source + target_weight*loss_surv_target

        dict_loss = coef_1 * (recon_loss_source + recon_loss_target) + coef_2 * (reg_term_source + reg_term_target) + coef_3*(norm_penalty + norm_penalty_A + ortho_penalty + ortho_penalty_b)

        #print('surv_loss', surv_loss)

        kernels = gaussian_kernel_tf(phiM_s, phiM_t, kernel_mul=2.0, kernel_num=5, fix_sigma=None)
        loss_ss = tf.reduce_sum(kernels[:batch_source.shape[0], :batch_source.shape[0]]) / (batch_source.shape[0] ** 2)
        loss_tt = tf.reduce_sum(kernels[batch_source.shape[0]:, batch_source.shape[0]:]) / (batch_target.shape[0] ** 2)
        loss_st = -2 * tf.reduce_sum(kernels[:batch_source.shape[0], batch_source.shape[0]:]) / (batch_source.shape[0] * batch_target.shape[0])
        
        min_adv_loss = min_coef*(loss_ss + loss_tt + loss_st)

        surv_dict_loss = (surv_loss + dict_loss + min_adv_loss)
    
        #print('surv_dict_loss', surv_dict_loss)
        
    variables_dict_class = (D.trainable_variables + A.trainable_variables + 
                            Bs.trainable_variables + Bt.trainable_variables + Ps.trainable_variables + Pt.trainable_variables + 
                            hc.trainable_variables + phiN.trainable_variables)

    grads_class_dict = tape.gradient(surv_dict_loss, variables_dict_class)
        
    optimizer_class_dict.apply_gradients(zip(grads_class_dict, variables_dict_class))

    unit_norm_clipper(A)
    unit_norm_clipper(D)

    
    with tf.GradientTape(persistent=True) as tape:

        D.trainable = True # reconstruc common
        A.trainable = False
        Bs.trainable = False
        Bt.trainable = False
        Ps.trainable = False
        Pt.trainable = False
        hc.trainable = False
        phiN.trainable = False
        phiM.trainable = True
        
        # ADVERSARIAL: MMD Loss DISCRIMINATOR ###########################################################

        # PsXs_ = Ps(batch_source)
        # PtXt_ = Pt(batch_target)

        ABsXs_ = A(Bs(batch_source))
        DABsXs_ = D(ABsXs_)

        # ABtXt_ = A(Bt(all_target_X))
        ABtXt_ = A(Bt(batch_target))
        DABtXt_ = D(ABtXt_)

        phiN_s = phiN(ABsXs_)
        phiM_s = phiM(phiN_s)

        phiN_t = phiN(ABtXt_)
        phiM_t = phiM(phiN_t)

        # hc_s = hc(phiN_s)
        # hc_t = hc(phiN_t)

        kernels = gaussian_kernel_tf(phiM_s, phiM_t, kernel_mul=2.0, kernel_num=5, fix_sigma=None)
        loss_ss = tf.reduce_sum(kernels[:batch_source.shape[0], :batch_source.shape[0]]) / (batch_source.shape[0] ** 2)
        loss_tt = tf.reduce_sum(kernels[batch_source.shape[0]:, batch_source.shape[0]:]) / (batch_target.shape[0] ** 2)
        loss_st = -2 * tf.reduce_sum(kernels[:batch_source.shape[0], batch_source.shape[0]:]) / (batch_source.shape[0] * batch_target.shape[0])
        
        adv_loss = loss_ss + loss_tt + loss_st

        max_adv_loss = -max_coef*(adv_loss)

    grads_phiM = tape.gradient(max_adv_loss, phiM.trainable_variables + D.trainable_variables)

    optimizer_phiM.apply_gradients(zip(grads_phiM, phiM.trainable_variables))

    return {
    "surv_dict_loss": surv_dict_loss.numpy(), # survival + dictionary loss
    "surv_loss": surv_loss.numpy(), # source + target survival loss
    "dict_loss": dict_loss.numpy(), # source + target dictionary loss
    "recon_loss_source": recon_loss_source.numpy(), # source dictionary loss
    "recon_loss_target": recon_loss_target.numpy(), # target dictionary loss
    "loss_surv_source": (source_weight*loss_surv_source).numpy(), # source survival loss
    "loss_surv_target": (target_weight*loss_surv_target).numpy(), # target survival loss
    "max_adv_loss":max_adv_loss.numpy(), # adversarial max loss
    "min_adv_loss": min_adv_loss.numpy()} # adversarial min loss

In [27]:
def run_model(X_source, y_source, time_source,
              X_train, y_train, time_target_train, 
              X_test, y_test, time_target_test,
              D, A, Bs, Bt, Ps, Pt, phiN, phiM, hc,
              X_target_oversampled, y_target_oversampled, time_target_oversampled,
              source_weight = 1.5,
              target_weight = 1.5,
              coef_1 = 2e1, # Mesma ordem de grandeza
              coef_2 = 1e-8,
              coef_3 = 1e-8,
              learning_rate_dict=0.01, momentum_dict=0.99,
              max_coef = 1e1, learning_rate_M=0.01, momentum_M=0.99,
              min_coef = 1e1, learning_rate_N=0.01, momentum_N=0.99,
              epochs = 10000, batch_size = 1024, patience = 100, min_epochs = 2000, delta=1e-5,
              weights_save_dir=None):
    
    # Initialize lists for tracking performance
    total_loss_history = []
    total_surv_history = []
    total_dict_history = []
    
    total_surv_loss_target_train = []
    total_surv_loss_source_train = []
    
    total_dict_loss_target_train = []
    total_dict_loss_source_train = []
    
    total_adv_min_loss_target_train = []
    total_adv_max_loss_source_train = []
    
    total_adv_loss_history = []
    
    total_adv_loss = 0
    total_loss = 0
    
    cindex_source_list = []
    cindex_train_list = []
    cindex_test_list = []
    
    print_epoch = False
    
    epochs = epochs
    batch_size = batch_size
    
    # Track best validation metrics
    best_cindex = 0.0
    best_val_loss = float('inf')  # Track best validation loss
    best_model_weights = None  # Store best model weights
    patience = patience #100 ; Number of epochs without improvement before stopping
    patience_counter = 0  # Track consecutive epochs without improvement
    min_epochs = min_epochs #1200


    best_source_loss = 0
    best_train_loss = 0
    
    best_val_loss = 0
    epoch_best = 0
    best_train_cindex = 0
    best_source_cindex = 0
    
    window_size = 40
    slope_window_size = 40
    loss_window = []
    slope_change_window = []  
    
    delta = delta #1e-5
    
    print('\nModel started to run !')
    print(" ")
    
    for epoch in range(epochs):

        #print("Epoch Running")
        
        total_surv_loss = 0
        total_dict_loss = 0
    
        loss_surv_target_train = 0
        loss_surv_source_train = 0
        
        loss_dict_target_train = 0
        loss_dict_source_train = 0
    
        total_adv_min = 0
        total_adv_max = 0
    
        batch_generator = StratifiedBatchGenerator(
        X_source, y_source, time_source,
        X_target_oversampled, y_target_oversampled, time_target_oversampled,
        batch_size=batch_size)
    
        for (batch_source, y_batch_source, time_batch_source,
            batch_target, y_batch_target, time_batch_target) in batch_generator:
    
            # unique, counts = np.unique(y_batch_target, return_counts=True)
            # print("Class counts (including NaNs):")
            # for cls, count in zip(unique, counts):
            #     print(f"  {cls}: {count}")
    
            all_target_X = tf.concat([X_train, X_test], axis=0)
            
            with tf.device('/GPU:0'):
                loss_values = pipeline_MMD(
                    batch_source, batch_target, y_batch_source, y_batch_target, all_target_X,
                    D, A, Bs, Bt, Ps, Pt, phiN, phiM, hc, 
                    time_batch_source, time_batch_target, 
                    source_weight = source_weight,
                    target_weight = target_weight,
                    coef_1 = coef_1, # Mesma ordem de grandeza
                    coef_2 = coef_2,
                    coef_3 = coef_3,
                    learning_rate_dict=learning_rate_dict, momentum_dict=momentum_dict,
                    max_coef = max_coef, learning_rate_M=learning_rate_M, momentum_M=momentum_M,
                    min_coef = min_coef, learning_rate_N=learning_rate_N, momentum_N=momentum_N)
                
            #print(' ')
            #print(' ')
            
            total_surv_loss += loss_values["surv_loss"] # source + target survival loss
            total_dict_loss += loss_values["dict_loss"] # source + target dictionary loss
    
            loss_surv_source_train += loss_values["loss_surv_source"] # source survival loss
            loss_surv_target_train += loss_values["loss_surv_target"] # target survival loss
            
            loss_dict_source_train += loss_values["recon_loss_source"] # source dictionary loss
            loss_dict_target_train += loss_values["recon_loss_target"] # target dictionary loss
    
            total_adv_max += loss_values["max_adv_loss"] # max adversarial loss
            total_adv_min += loss_values["min_adv_loss"] # min adversarial loss
    
        total_adv_loss = total_adv_min - total_adv_max  #we put minus since the negative sign is just for maximization purposes
        
        total_loss = total_surv_loss + total_dict_loss + total_adv_loss
        
        # Track loss history
        total_loss_history.append(total_loss)
        total_surv_history.append(total_surv_loss)
        total_dict_history.append(total_dict_loss)
        total_adv_loss_history.append(total_adv_loss)
    
        total_surv_loss_target_train.append(loss_surv_target_train)
        total_surv_loss_source_train.append(loss_surv_source_train)
    
        total_dict_loss_target_train.append(loss_dict_target_train)
        total_dict_loss_source_train.append(loss_dict_source_train)
    
        total_adv_min_loss_target_train.append(total_adv_min)
        total_adv_max_loss_source_train.append(-total_adv_max)
    
        def classifier_(X_input, Bt, A, phiN, hc):
            input_ = Input(shape=(X_input.shape[1],), name="_input")
            logits = hc(phiN(A(Bt(input_))))
            model_ = Model(inputs=input_, outputs=logits, name='Survival Model')
            return model_
    
        eval_ = classifier_(X_test, Bt, A, phiN, hc)
        eval_source = classifier_(X_source, Bs, A, phiN, hc)
    
        #print(eval_.summary())
        #print(eval_source.summary())
    
        if print_epoch:
            #Print loss for the current epoch
            print(f"Epoch {epoch + 1} ########################")
            print(" ")
            print(f"Total Loss (Survival + Dict + Adv): {total_loss:.4f}")
            print(" ")
            print(f"Total Dictionary Loss: {total_dict_loss:.4f}")
            print(f">>Total Source Dictionary Loss: {loss_dict_source_train:.4f}")
            print(f">>Total Target Dictionary Loss: {loss_dict_target_train:.4f}")
            print(" ")
            print(f"Total Survival Loss: {total_surv_loss:.4f}")
            print(f">>Total Source Survival Loss: {loss_surv_source_train:.4f}")
            print(f">>Total Target Survival Loss: {loss_surv_target_train:.4f}")
            print(" ")
            print(f"Total Adversarial Loss: {total_adv_loss:.4f}")
            print(f">>Total MIN Adversarial Loss: {total_adv_min:.4f}")
            print(f">>Total MAX Adversarial Loss: {-total_adv_max:.4f}")
            print(" ")
        
        # Evaluate model performance
        cindex_source = survival_metrics(eval_source, X_source, time_source, y_source, data_type='source_training')
        cindex_target_train = survival_metrics(eval_, X_train, time_target_train, y_train, data_type='target_training')
        cindex_target_test = survival_metrics(eval_, X_test, time_target_test, y_test, data_type='target_testing')
        
        if print_epoch:
            print(f"Source - Concordance Index (C-index): {cindex_source:.4f}")
            print(f"Target Train - Concordance Index (C-index): {cindex_target_train:.4f}")
            print(f"Target Test - Concordance Index (C-index): {cindex_target_test:.4f}")
            print(" ")
        
        cindex_source_list.append(cindex_source)
        cindex_train_list.append(cindex_target_train)
        cindex_test_list.append(cindex_target_test)
    
        # val_loss = cox_ph_loss_batch(y_test, time_target_test, eval_(X_test))
        # class_test_history.append(val_loss)
        # print(f'Target testing - Total Survival loss: {val_loss:.4f}')
        
        #if total_loss is not None and isinstance(total_loss, (float, int)) and np.isfinite(total_loss):
        #print(total_loss)
        loss_window.append(total_loss)
    
        if len(loss_window) > window_size:
            loss_window.pop(0)
    
        if len(loss_window) == window_size:
            # Fit a linear regression to the recent loss values
            x = np.arange(window_size).reshape(-1, 1)
            y = np.array(loss_window).reshape(-1, 1)
            slope_change = LinearRegression().fit(x, y).coef_[0][0]
    
            slope_change_window.append(slope_change)
    
            if len(slope_change_window) > slope_window_size:
                slope_change_window.pop(0)
    
            if len(slope_change_window) == slope_window_size:
                mean_change = np.mean(slope_change_window)
                
                if mean_change <= delta and epoch + 1 > min_epochs:
                    patience_counter += 1
    
                    if patience_counter >= patience*0.98:

                        if cindex_target_test > best_cindex:
                        
                            # print('entering early stopping')
                                
                            best_source_loss = loss_surv_source_train
                            best_train_loss = loss_surv_target_train
                            
                            best_val_loss = cox_ph_loss_batch(y_test, time_target_test, eval_(X_test))
                            epoch_best = epoch + 1
                            best_cindex = cindex_target_test
                            best_train_cindex = cindex_target_train
                            best_source_cindex = cindex_source
                            
                            best_model_weights = (
                                eval_.get_weights(),
                                eval_source.get_weights()
                            )
                            
                            save_weights(eval_, 'target', weights_save_dir)
                            save_weights(eval_source, 'source', weights_save_dir)
        
                            #print(" ")
                            print(f"Epoch {epoch + 1} - Best Testing C-index: {best_cindex:.4f} with survival testing loss {best_val_loss:.4f}")
                            print(f"Best Source C-index: {best_source_cindex:.4f}")
                            print(f"Best Training Target C-index: {best_train_cindex:.4f}")
                else:
                    patience_counter = 0
    
    
        if patience_counter >= patience and epoch + 1 > min_epochs:
            print(f"\nEarly stopping: loss change < {delta} for {patience} consecutive epochs.")
        #    print("Restoring best model weights...")
            break
            
    # Restore the best model after early stopping
    if best_model_weights is not None:
        eval_.set_weights(best_model_weights[0])
        eval_source.set_weights(best_model_weights[1])
        print(" ")
        print(f"Restored best model with lowest Survival Loss: {best_val_loss:.4f} and Best C-index: {best_cindex:.4f}")
    
    print(" ")
    print('Model finished to run !')

    return {
    "best_val_loss": best_val_loss,
    "best_source_loss": best_source_loss,
    "best_train_loss": best_train_loss,
    "best_cindex": best_cindex,
    "best_train_cindex": best_train_cindex,
    "best_source_cindex": best_source_cindex,
    "cindex_source_list": cindex_source_list,
    "cindex_train_list": cindex_train_list,
    "cindex_test_list": cindex_test_list,
    "total_loss_history": total_loss_history,
    "total_surv_history": total_surv_history,
    "total_dict_history": total_dict_history,
    "total_surv_loss_source_train": total_surv_loss_source_train,
    "total_surv_loss_target_train": total_surv_loss_target_train,
    "total_dict_loss_target_train":total_dict_loss_target_train,
    "total_dict_loss_source_train": total_dict_loss_source_train,
    "total_adv_min_loss_target_train":total_adv_min_loss_target_train,
    "total_adv_max_loss_source_train" :total_adv_max_loss_source_train,
    "total_adv_loss_history": total_adv_loss_history,
    "epoch_best": epoch_best
    }, eval_, eval_source

## Model Plots

In [28]:
def total_convergence_plot(total_loss_history, total_surv_history, total_dict_history, total_adv_loss_history, save_dir):
    
    # Plot each log-transformed loss
    plt.plot(range(1, len(total_loss_history) + 1), total_loss_history, marker='o', color='r', label='Total Loss', markersize=1, linewidth=2, alpha=1)
    plt.plot(range(1, len(total_surv_history) + 1), total_surv_history, color='g', label='Total Survival Loss', linewidth=1.5, alpha=0.6)
    plt.plot(range(1, len(total_dict_history) + 1), total_dict_history, color='c', label='Total Dictionary Loss', linewidth=1.5, alpha=0.6)
    plt.plot(range(1, len(total_adv_loss_history) + 1), total_adv_loss_history, color='orange', label='Total Adversarial Loss', linewidth=1.5, alpha=0.6)
    
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'Total Training Loss Over Epochs - SGD optimizer')
    plt.grid(True, alpha=0.3)
    #plt.ylim(-5, 65)  
    plt.legend(loc='best')
    #plt.show()
    plt.savefig(os.path.join(save_dir, "total_loss.png"), dpi=300, bbox_inches="tight")
    plt.close()

def convergence_plot(total_surv_history, total_dict_history, total_adv_loss_history, save_dir):
    
    plt.plot(range(1, len(total_surv_history) + 1), total_surv_history, color='g', label='Total Survival Loss', linewidth=2, alpha=1)
    plt.plot(range(1, len(total_dict_history) + 1), total_dict_history, color='c', label='Total Dictionary Loss', linewidth=2, alpha=1)
    plt.plot(range(1, len(total_adv_loss_history) + 1), total_adv_loss_history, color='orange', label='Total Adversarial Loss', linewidth=2, alpha=1)
    
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'Total Training Loss Over Epochs - SGD optimizer')
    plt.grid(True, alpha=0.3)
    #plt.ylim(-5, 65)  
    plt.legend(loc='best')
    #plt.show()
    plt.savefig(os.path.join(save_dir, "separate_loss.png"), dpi=300, bbox_inches="tight")
    plt.close()

def surv_loss_plot(total_surv_loss_source_train, total_surv_loss_target_train, save_dir):
    
    plt.plot(range(1, len(total_surv_loss_source_train) + 1), total_surv_loss_source_train, marker='o', color='g', label='Total Source Survival Loss', markersize=0.6, linewidth=2, alpha=1)
    plt.plot(range(1, len(total_surv_loss_target_train) + 1), total_surv_loss_target_train, marker='o', color='c', label='Total Target Survival Loss', markersize=0.6, linewidth=2, alpha=1)
    
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'Training Survival Loss Over Epochs - SGD optimizer')
    plt.grid(True, alpha=0.3)
    #plt.ylim(-5, 65)  
    plt.legend(loc='best')
    #plt.show()
    plt.savefig(os.path.join(save_dir, "surv_loss.png"), dpi=300, bbox_inches="tight")
    plt.close()

def dict_loss_plot(total_dict_loss_source_train, total_dict_loss_target_train, save_dir):
    
    plt.plot(range(1, len(total_dict_loss_source_train) + 1), total_dict_loss_source_train, marker='o', color='g', label='Total Source Dictionary Loss', markersize=0.6, linewidth=2, alpha=1)
    plt.plot(range(1, len(total_dict_loss_target_train) + 1), total_dict_loss_target_train, marker='o', color='c', label='Total Target Dictionary Loss', markersize=0.6, linewidth=2, alpha=1)
    
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'Training Dictionary Loss Over Epochs - SGD optimizer')
    plt.grid(True, alpha=0.3)
    #plt.ylim(-5, 65)  
    plt.legend(loc='best')
    plt.savefig(os.path.join(save_dir, "dict_loss.png"), dpi=300, bbox_inches="tight")
    plt.close()
    #plt.show()

def cindex_plot(cindex_source_list, cindex_train_list, cindex_test_list, save_dir):
    
    plt.plot(range(1, len(cindex_source_list) + 1), cindex_source_list, color='g', label=f'C-index source', linewidth=2, alpha=0.7, marker='o', markersize=0.6)
    plt.plot(range(1, len(cindex_train_list) + 1), cindex_train_list, color='b', label=f'C-index train target', linewidth=2, alpha=0.7, marker='o', markersize=0.6)
    plt.plot(range(1, len(cindex_test_list) + 1), cindex_test_list, color='m', label=f'C-index test target', linewidth=2, alpha=0.7, marker='o', markersize=0.6)
    
    plt.xlabel('Epochs')
    plt.ylabel('C-index')
    plt.title(f'C-index Over Epochs')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    #plt.show()
    plt.savefig(os.path.join(save_dir, "cindex_loss.png"), dpi=300, bbox_inches="tight")
    plt.close()

## Model Performance Evaluation

In [29]:
def mmd_loss(batch_source, batch_target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):

        batch_source, batch_target = tf.convert_to_tensor(batch_source, dtype=tf.float32), tf.convert_to_tensor(batch_target, dtype=tf.float32)

        kernels = gaussian_kernel_tf(batch_source, batch_target, kernel_mul=kernel_mul, kernel_num=kernel_num, fix_sigma=fix_sigma)
        loss_ss = tf.reduce_sum(kernels[:batch_source.shape[0], :batch_source.shape[0]]) / (batch_source.shape[0] ** 2)
        loss_tt = tf.reduce_sum(kernels[batch_source.shape[0]:, batch_source.shape[0]:]) / (batch_target.shape[0] ** 2)
        loss_st = -2 * tf.reduce_sum(kernels[:batch_source.shape[0], batch_source.shape[0]:]) / (batch_source.shape[0] * batch_target.shape[0])
        
        loss = loss_ss + loss_tt + loss_st

        return loss

In [30]:
def predict_survival_function(model, X_, times, events):
    """
    Compute survival functions using a CoxPH model trained with SGD.

    Parameters
    ----------
    model : trained TF/PyTorch model
        Must output log-risk scores (x^T beta).
    X_train : np.ndarray
        Training covariates (n_train, d).
    times : np.ndarray
        Training durations (n_train,).
    events : np.ndarray
        Training event indicators (1=event, 0=censor).
    X_test : np.ndarray
        Test covariates (n_test, d).

    Returns
    -------
    surv_df : pd.DataFrame
        Survival functions. Rows = unique times, Columns = patients in X_test.
    """

    times = np.asarray(times).squeeze()
    events = events
    log_risk_scores_test = model(X_).numpy().squeeze()
    
    # Step 2: Breslow baseline cumulative hazard
    unique_times = np.sort(np.unique(times))
    baseline_hazard = []
    for t in unique_times:
        at_risk = log_risk_scores_test[times >= t]
        observed = (times == t) & (events == 1)
        numerator = np.sum(observed)
        denominator = np.sum(np.exp(at_risk))
        h_t = numerator / (denominator + 1e-8)
        baseline_hazard.append(h_t)
    baseline_H = np.cumsum(baseline_hazard)
    baseline_S = np.exp(-baseline_H)

    # Step 3: Patient-specific survival curves
    surv_dict = {}
    for i, log_r in enumerate(log_risk_scores_test):
        surv_dict[i] = baseline_S ** np.exp(log_r)

    surv_df = pd.DataFrame(surv_dict, index=unique_times)
    return surv_df


In [31]:
def metrics_output(model, model_source,
                   X_train, time_train, event_train, 
                   X_test, time_test, event_test,
                   X_source, time_source, event_source,
                   unlabeled_fraction, save_dir):


    # Concordance
    def cindex_value(evl):
        
        cindex, _, _ = evl.concordance()
        if cindex < 0.5:
            cindex = 1 - cindex
        return cindex

    def ibs(evl):
        
        # Integrated Brier Score (only if fully labeled)
        
        ibs = None
        if unlabeled_fraction != 1.0:
            ibs = evl.integrated_brier_score(num_points=None, IPCW_weighted=True, draw_figure=False)

        return ibs

    def valid_train(X_train, time_train, event_train):
        # Cast to float so TensorFlow can evaluate NaN
        event_train = tf.cast(event_train, tf.float32)
        time_train = tf.cast(time_train, tf.float32)
        
        valid_mask_target = ~tf.math.is_nan(event_train) & ~tf.math.is_nan(time_train)
        
        if tf.reduce_any(valid_mask_target):
            y_target_valid = tf.boolean_mask(event_train, valid_mask_target).numpy()
            t_target_valid = tf.boolean_mask(time_train, valid_mask_target).numpy()
            X_valid = tf.boolean_mask(X_train, valid_mask_target).numpy()
        else:
            y_target_valid, t_target_valid = np.array([]), np.array([])
        
        return X_valid, t_target_valid, y_target_valid


    if unlabeled_fraction != 1.0:

        X_train, time_train, event_train = valid_train(X_train, time_train, event_train.reshape(-1))

        isd_train = predict_survival_function(model, X_train, time_train, event_train.reshape(-1))
        isd_source = predict_survival_function(model_source, X_source, time_source, event_source.reshape(-1))
        isd_test = predict_survival_function(model, X_test, time_test, event_test.reshape(-1))
    
        evl = LifelinesEvaluator(isd_test, time_test, event_test.reshape(-1),
                                 time_train, event_train.reshape(-1))

        evl_train = LifelinesEvaluator(isd_train, time_train, event_train.reshape(-1))

        evl_source = LifelinesEvaluator(isd_source, time_source, event_source.reshape(-1))

        cindex_train = cindex_value(evl_train)
        cindex_test = cindex_value(evl)
        cindex_source = cindex_value(evl_source)

        mse_train = evl_train.mse(method="Hinge", weighted=False, log_scale=True)
        mse_test = evl.mse(method="Hinge", weighted=False, log_scale=True)
        mse_source = evl_source.mse(method="Hinge", weighted=False, log_scale=True)

        ibs_test = ibs(evl)

        # Collect metrics in dictionary
        metrics = {
            "Unlabeled_fraction": unlabeled_fraction, 
            "C-index Source": np.round(cindex_source, 4),
            "C-index Train": np.round(cindex_train, 4),
            "C-index Test": np.round(cindex_test, 4),
            "IBS Test": np.round(ibs_test,4),
            "MSE Source": np.round(mse_source, 4),
            "MSE Train": np.round(mse_train, 4),
            "MSE Test": np.round(mse_test, 4)
        }
            
        
    if unlabeled_fraction == 1.0:
        
        isd_source = predict_survival_function(model_source, X_source, time_source, event_source.reshape(-1))
        isd_test = predict_survival_function(model, X_test, time_test, event_test.reshape(-1))

        evl = LifelinesEvaluator(isd_test, time_test, event_test.reshape(-1))
        evl_source = LifelinesEvaluator(isd_source, time_source, event_source.reshape(-1))

        cindex_test = cindex_value(evl)
        cindex_source = cindex_value(evl_source)

        mse_test = evl.mse(method="Hinge", weighted=False, log_scale=True)
        mse_source = evl_source.mse(method="Hinge", weighted=False, log_scale=True)


        # Collect metrics in dictionary
        metrics = {
            "Unlabeled_fraction": unlabeled_fraction,
            "C-index Source": np.round(cindex_source, 4),
            "C-index Train": None,
            "C-index Test": np.round(cindex_test, 4),
            "IBS Test": None,
            "MSE Source": np.round(mse_source, 4),
            "MSE Train": None,
            "MSE Test": np.round(mse_test, 4)
        }
            

    # # Convert to DataFrame for clean printing
    # df_metrics = pd.DataFrame(metrics, index=[0])
    # print("\n📊 Evaluation Metrics\n", df_metrics.T)

    print(" ")

    df = pd.DataFrame([metrics])   

    return metrics

## Visualization

In [32]:
def project_pca(Xs, Xt, n_components=2):
    """
    Project source and target into the same PCA space.
    """
    X_concat = np.vstack([Xs, Xt])
    pca = PCA(n_components=n_components)
    X_proj = pca.fit_transform(X_concat)
    Xs_proj = X_proj[:len(Xs)]
    Xt_proj = X_proj[len(Xs):]
    return Xs_proj, Xt_proj

def plot_comparison(Xs1, Xt1, Xs2, Xt2, Xs_dict, Xt_dict, title1="Before Projection", title2="After Dictionary Learning", title3="After MMD", save_dir=None):
    """
    Compare two sets of source/target projections with fixed axis limits.
    """
    # Compute global axis limits
    all_x = np.concatenate([Xs1[:,0], Xt1[:,0], Xs2[:,0], Xt2[:,0], Xs_dict[:,0], Xt_dict[:,0]])
    all_y = np.concatenate([Xs1[:,1], Xt1[:,1], Xs2[:,1], Xt2[:,1], Xs_dict[:,1], Xt_dict[:,1]])
    x_min, x_max = all_x.min() - 0.5, all_x.max() + 0.1
    y_min, y_max = all_y.min() - 0.5, all_y.max() + 0.1

    fig, axes = plt.subplots(3, 1, figsize=(6,15))

    # First plot
    axes[0].scatter(Xs1[:,0], Xs1[:,1], c="tab:blue", alpha=0.6, label="Source")
    axes[0].scatter(Xt1[:,0], Xt1[:,1], c="tab:orange", alpha=0.6, label="Target")
    axes[0].set_xlim(x_min, x_max)
    axes[0].set_ylim(y_min, y_max)
    axes[0].set_title(title1)
    axes[0].set_xlabel("PCA 1")
    axes[0].set_ylabel("PCA 2")
    axes[0].legend()

    # Second plot
    axes[1].scatter(Xs_dict[:,0], Xs_dict[:,1], c="tab:blue", alpha=0.6, label="Source")
    axes[1].scatter(Xt_dict[:,0], Xt_dict[:,1], c="tab:orange", alpha=0.6, label="Target")
    axes[1].set_xlim(x_min, x_max)
    axes[1].set_ylim(y_min, y_max)
    axes[1].set_title(title2)
    axes[1].set_xlabel("PCA 1")
    axes[1].set_ylabel("PCA 2")
    axes[1].legend()


    axes[2].scatter(Xs2[:,0], Xs2[:,1], c="tab:blue", alpha=0.6, label="Source")
    axes[2].scatter(Xt2[:,0], Xt2[:,1], c="tab:orange", alpha=0.6, label="Target")
    axes[2].set_xlim(x_min, x_max)
    axes[2].set_ylim(y_min, y_max)
    axes[2].set_title(title3)
    axes[2].set_xlabel("PCA 1")
    axes[2].set_ylabel("PCA 2")
    axes[2].legend()

    plt.tight_layout()
    # plt.show()
    plt.savefig(os.path.join(save_dir, "pca_data_view.png"), dpi=300, bbox_inches="tight")
    plt.close()

In [33]:
def dict_models(model, X):
    new_input = Input(shape=(X.shape[1],), name="mmd_input")

    x = new_input
    for layer in model.layers[1:-2]:  # skip Input and last layer
        x = layer(x)

    truncated_model = Model(
        inputs=new_input,   
        outputs=x,
        name="mmd_model"
    )
    return truncated_model

In [34]:
def mmd_models(model, X):
    new_input = Input(shape=(X.shape[1],), name="mmd_input")

    x = new_input
    for layer in model.layers[1:-1]:  # skip Input and last layer
        x = layer(x)

    truncated_model = Model(
        inputs=new_input,   # ✅ use the new input
        outputs=x,
        name="mmd_model"
    )
    return truncated_model

## Run model

In [35]:
from itertools import product

In [36]:
n_source=1000
n_target=100
seed_source=8
seed_target=7
marg_source=1.0
marg_target=0.9
cond_source=1.0
cond_target=1.0
censor_source= 0.2
censor_target= 0.2
d_features=15
shift_type='marginal'

In [37]:
unlabeled_fraction_list = [0.0]

hc_reg_strength_list = [1e-3, 1e-1]  
reg_strength_list = [0]  

same_unit_list = [16, 64, 128, 256] 
A_unit_list = [16, 64, 128, 256]
hc_unit_list = [[], [16]]

source_weight_list = [0.5]
target_weight_list = [1]

coef1_list = [2.5]
learning_rate_dict_list = [0.01]

max_coef_list = [2.5]
learning_rate_M_list = [0.01]

min_coef_list = [2.5]
learning_rate_N_list = [0.01]

epochs_list = [10000] #10000
patience_list = [50] #50
min_epochs_list = [400] #400

delta_list = [1e-4] #1e-4


# --- Count total runs ---
grid = list(product(
    unlabeled_fraction_list,
    same_unit_list,
    hc_unit_list,
    A_unit_list,
    hc_reg_strength_list,
    reg_strength_list,
    target_weight_list,
    source_weight_list,
    coef1_list,
    learning_rate_dict_list,
    max_coef_list,
    learning_rate_M_list,
    min_coef_list,
    learning_rate_N_list,
    epochs_list,
    patience_list,
    min_epochs_list,
    delta_list
))

total_runs = len(grid)
print(f"Total possible runs: {total_runs}")

# --- Create base folder for experiments ---
base_dir = "experiments"
os.makedirs(base_dir, exist_ok=True)

csv_path = os.path.join(base_dir, "experiment_log.csv")

# --- Initialize CSV if not exists ---
if not os.path.exists(csv_path):
    df_init = pd.DataFrame(columns=[
        # run tracking
        "run_id", "run_dir", "fold",

        # dataset params
        "n_source", "n_target", "seed_source", "seed_target",
        "marg_source", "marg_target", "cond_source", "cond_target",
        "censor_source", "censor_target", "d_features", "shift_type",

        # dataset stats
        "X_source_shape", "y_source_shape", "Time_source_min", "Time_source_max",
        "X_target_shape", "y_target_shape", "Time_target_min", "Time_target_max",

        # hyperparams
        "unlabeled_fraction", "same_unit", "hc_unit", "A_unit", "hc_reg_strength", "reg_strength",
        "target_weight", "source_weight", "coef1", "learning_rate_dict",
        "max_coef", "learning_rate_M", "min_coef", "learning_rate_N",
        "epochs", "patience", "min_epochs", "delta",

        # metrics
        "Empty", "C-index Source", "C-index Train", "C-index Test",
        "IBS Test", "MSE Source", "MSE Train", "MSE Test"
    ])
    df_init.to_csv(csv_path, index=False)
    

Total possible runs: 64


In [59]:
Xt_original, _, _ = import_data(path="/kaggle/input/03082025-selected-tcga-for-da-data/mirna/df_black_kirp.csv", event="OS", time='OS.time')
Xs_original, _, _ = import_data(path="/kaggle/input/03082025-selected-tcga-for-da-data/mirna/df_white_kirp.csv", event="OS", time='OS.time')

In [60]:
Xt, yt, Tt = import_data(path="/kaggle/input/21112025-feature-select-aggr-3/white_black_kirp/black_kirp_target.csv", event="event", time='time_to_event')
Xs, ys, Ts = import_data(path="/kaggle/input/21112025-feature-select-aggr-3/white_black_kirp/white_kirp_source.csv", event="event", time='time_to_event')

Xt, yt, Tt = np.array(Xt), np.array(yt), np.array(Tt)
Xs, ys, Ts = np.array(Xs), np.array(ys), np.array(Ts)

In [61]:
print("Source Data Statistics: ")
print(f"X_source shape: {Xs.shape}")
print(f"y_source sum: {ys.sum()}")
print(f"min_max source time: {np.round(Ts.min(),2)}, {np.round(Ts.max(),2)}")
print(" ")
print("Target Data Statistics: ")
print(f"X_target shape: {Xt.shape}")
print(f"y_target sum: {yt.sum()}")
print(f"min_max source time: {np.round(Tt.min(),2)}, {np.round(Tt.max(),2)}")

X_source, y_source, time_source = Xs, ys, Ts

X_target, y_target, time_target = Xt, yt, Tt

Source Data Statistics: 
X_source shape: (202, 93)
y_source sum: 32.0
min_max source time: 0.0, 5925.0
 
Target Data Statistics: 
X_target shape: (61, 20)
y_target sum: 7.0
min_max source time: 3.0, 3035.0


In [62]:
X_source_shape = Xs.shape
y_source_shape = ys.shape
Time_source_min = Ts.min()
Time_source_max = Ts.max()

X_target_shape = Xt.shape
y_target_shape = yt.shape
Time_target_min = Tt.min()
Time_target_max = Tt.max()

In [63]:
for run_id, (unlabeled_fraction, same_unit, hc_unit, A_unit, hc_reg_strength, reg_strength,
             target_weight, source_weight, coef1, learning_rate_dict,
             max_coef, learning_rate_M, min_coef, learning_rate_N,
             epochs, patience, min_epochs, delta) in enumerate(grid, start=1):

    # Create folder for this run
    run_dir = os.path.join(base_dir, f"run_{run_id:03d}") 

    X_train, X_test, y_train, y_test, time_target_train, time_target_test = train_test_split(
        X_target,  y_target, time_target,
        test_size=0.15,
        stratify=yt,   # ensures event/censor ratio stays similar
        random_state=42
    )

    print("X_shape",X_train.shape,"X_test",y_test.shape)
    print("y_train_sum",y_train.sum(),"y_test_sum",y_test.sum())

    # --- K-fold CV on training set ---
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):

        fold_dir = os.path.join(run_dir, f"fold_{fold:03d}") 
        os.makedirs(fold_dir, exist_ok=True)
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        time_target = np.array(time_target)
        time_tr, time_val = time_target_train[train_idx], time_target_train[val_idx]

        print(f"\nFold {fold+1}: train={len(train_idx)}, val={len(val_idx)}")

        ## Setting unlabeled fraction and oversampling
        y_tr, time_tr = assign_unlabel(y_tr, time_tr, unlabeled_fraction=unlabeled_fraction)
        X_target_oversampled , y_target_oversampled, time_target_oversampled = oversample_target(X_tr, y_tr, time_tr, X_source.shape[0], seed=42)
    
    
        ## Setting network
        # params_(same_unit = 200, A_unit=200)
        # params_1layer(same_unit = 64, A_unit=200)
        # params_2layer(different_unit = 64, same_unit = 64, A_unit=200)
        
        # D, A, Bs, Bt, Ps, Pt, phiN, phiM = params_(same_unit=same_unit, A_unit=A_unit, l2_reg=reg_strength)
        D, A, Bs, Bt, Ps, Pt, phiN, phiM = params_1layer(same_unit = same_unit, A_unit = A_unit, l2_reg=reg_strength)
        
        hc, phi_config = build_surv_block(
            hidden_units=hc_unit,
            activation='relu',
            dropout_rate=0.3,
            reg_type='l2',
            reg_strength=hc_reg_strength,
            use_batchnorm=True,
            print_specs=False
        )
    
        ## Run model
        out, eval_, eval_source = run_model(X_source, y_source, time_source,
                                            X_tr, y_tr, time_tr, 
                                            X_val, y_val, time_val,
                                            D, A, Bs, Bt, Ps, Pt, phiN, phiM, hc,
                                            X_target_oversampled, y_target_oversampled, time_target_oversampled,
                                            source_weight = source_weight,
                                            target_weight = target_weight,
                                            coef_1 = coef1, # Mesma ordem de grandeza
                                            coef_2 = 1e-8,
                                            coef_3 = 1e-8,
                                            learning_rate_dict=learning_rate_dict, momentum_dict=0.99,
                                            max_coef = max_coef, learning_rate_M=learning_rate_M, momentum_M=0.99,
                                            min_coef = min_coef, learning_rate_N=learning_rate_N, momentum_N=0.99,
                                            epochs = epochs, batch_size = 1024, patience = patience, min_epochs = min_epochs, delta=delta_list,
                                            weights_save_dir = fold_dir)
    
        ## Metrics output
        metrics= metrics_output(eval_, eval_source,
                                X_tr, time_tr, y_tr, 
                                X_val, time_val, y_val,
                                X_source, time_source, y_source,
                                unlabeled_fraction=unlabeled_fraction, 
                                save_dir=fold_dir)
    
        # Extract from dictionary
        total_loss_history              = out["total_loss_history"]
        total_surv_history              = out["total_surv_history"]
        total_dict_history              = out["total_dict_history"]
        total_adv_loss_history          = out["total_adv_loss_history"]
        
        total_surv_loss_source_train    = out["total_surv_loss_source_train"]
        total_surv_loss_target_train    = out["total_surv_loss_target_train"]
        
        total_dict_loss_source_train    = out["total_dict_loss_source_train"]
        total_dict_loss_target_train    = out["total_dict_loss_target_train"]
        
        total_adv_min_loss_target_train = out["total_adv_min_loss_target_train"]
        total_adv_max_loss_source_train = out["total_adv_max_loss_source_train"]
        
        cindex_source_list              = out["cindex_source_list"]
        cindex_train_list               = out["cindex_train_list"]
        cindex_test_list                = out["cindex_test_list"]
        
        ## Plotting images
        total_convergence_plot(total_loss_history, total_surv_history, total_dict_history, total_adv_loss_history, fold_dir)
        convergence_plot(total_surv_history, total_dict_history, total_adv_loss_history, fold_dir)
        surv_loss_plot(total_surv_loss_source_train, total_surv_loss_target_train, fold_dir)
        dict_loss_plot(total_dict_loss_source_train, total_dict_loss_target_train, fold_dir)
        cindex_plot(cindex_source_list, cindex_train_list, cindex_test_list, fold_dir)
    
        #PCA plots
        proj_source = mmd_models(eval_source, X_source)
        proj_source_d = dict_models(eval_source, X_source)
    
        proj_ = mmd_models(eval_, X_target)
        proj_d = dict_models(eval_, X_target)
        
        Xs_proj_before, Xt_proj_before = project_pca(Xs_original, Xt_original)
        Xs_proj_after_d, Xt_proj_after_d = project_pca(proj_source_d(X_source), proj_d(X_target))
        Xs_proj_after, Xt_proj_after = project_pca(proj_source(X_source), proj_(X_target))
        plot_comparison(Xs_proj_before, Xt_proj_before, Xs_proj_after, Xt_proj_after, Xs_proj_after_d, Xt_proj_after_d, save_dir=fold_dir)
    
    
        # Save everything as one row in CSV
        row = {
            "run_id": run_id,
            "run_dir": run_dir,
            "fold": fold+1,
    
            # dataset params
            "n_source": n_source,
            "n_target": n_target,
            "seed_source": seed_source,
            "seed_target": seed_target,
            "marg_source": marg_source,
            "marg_target": marg_target,
            "cond_source": cond_source,
            "cond_target": cond_target,
            "censor_source": censor_source,
            "censor_target": censor_target,
            "d_features": d_features,
            "shift_type": shift_type,
    
            # dataset stats
            "X_source_shape": X_source_shape,
            "y_source_shape": y_source_shape,
            "Time_source_min": Time_source_min,
            "Time_source_max": Time_source_max,
            "X_target_shape": X_target_shape,
            "y_target_shape": y_target_shape,
            "Time_target_min": Time_target_min,
            "Time_target_max": Time_target_max,
    
            # hyperparams
            "unlabeled_fraction": unlabeled_fraction,
            "same_unit": same_unit,
            "hc_unit": hc_unit, 
            "A_unit": A_unit,
            "hc_reg_strength": hc_reg_strength,
            "reg_strength": reg_strength,
            "target_weight": target_weight,
            "source_weight": source_weight,
            "coef1": coef1,
            "learning_rate_dict": learning_rate_dict,
            "max_coef": max_coef,
            "learning_rate_M": learning_rate_M,
            "min_coef": min_coef,
            "learning_rate_N": learning_rate_N,
            "epochs": epochs,
            "patience": patience,
            "min_epochs": min_epochs,
            "delta": delta
        }
    
        # merge metrics in
        row.update(metrics)
    
        # append to CSV
        df = pd.DataFrame([row])
        df.to_csv(csv_path, mode="a", header=False, index=False)
    
    print(f"Logged run {run_id}/{total_runs} with metrics to {csv_path}")

X_shape (51, 20) X_test (10,)
y_train_sum 6.0 y_test_sum 1.0

Fold 1: train=40, val=11

Number of unlabeled samples: 0
Number of labeled samples: 40

Model started to run !
 


KeyboardInterrupt: 